# Annotate OMERO images with micro-SAM

Pick images from an OMERO dataset, project, plate or screen, annotate them in napari with micro-SAM, and upload the masks back to OMERO as ROIs. Each **training set** has a tracking table on OMERO and a local folder, so you can stop at any time and continue later; anything a crash left half-done is repaired when you open the set again.

Run the cells from top to bottom.

## 1. Import

In [1]:
import omero_annotate_ai
from omero_annotate_ai import create_omero_connection_widget, create_workflow_widget

print(f"omero-annotate-ai {omero_annotate_ai.__version__}")

omero-annotate-ai 0.5.2


## 2. Connect to OMERO

Fill in the server and your credentials, then click **Connect**.

In [2]:
conn_widget = create_omero_connection_widget()
conn_widget.display()

In [4]:
conn = conn_widget.get_connection()
if conn is None:
    raise ConnectionError(
        "Not connected yet: use the widget above, then rerun this cell."
    )
print("Connected to OMERO.")

Connected to OMERO.


## 3. Create or open a training set

1. **Data**: choose the container type and the container.
2. **New training set**: name, channel, timepoint/z-slice, 3D or patches, images per split, seed (empty = random) and storage (*OMERO* uploads masks and keeps a tracking table; *Local folder only* writes nothing to OMERO).
   **Open training set**: pick an earlier set; its summary shows progress and the notes of earlier sessions. Optionally add images per split, and choose *Annotate pending* or *Review annotated* (opens annotated images with their saved masks to correct them).
3. **Annotation**: model and batch size (images per napari session, 0 = all), then click **Create** / **Open**.

In [5]:
workflow_widget = create_workflow_widget(connection=conn)
workflow_widget.display()

### Optional: settings the widget does not show

For a *new* set you can adjust the config before creating it, e.g. well filters or context channels shown next to the annotated channel in napari.

In [6]:
# from omero_annotate_ai import TrainingSet
# config = workflow_widget.get_config()
# config.omero.well_filters = {"cellline": ["U2OS"], "treatment": ["Control"]}
# config.spatial_coverage.context_channels = [0, 2]
# ts = TrainingSet.create(config, conn, events=workflow_widget.events)

## 4. Annotate

napari opens with micro-SAM for each batch. Annotate the images and close napari to save them; the masks go to OMERO (or only to the local folder) and progress appears in the widget. Add a note about this session, e.g. what you did or skipped. With *Review annotated* chosen, the annotated images open with their saved masks instead; changed masks replace the ones on OMERO.

In [ ]:
ts = workflow_widget.training_set()
saved = workflow_widget.annotate(
    note=None
)  # e.g. note="nuclei only, small debris skipped"

print(f"Saved {saved} units in {ts.name}; local folder: {ts.bundle_dir}")

After the session you can still change its note in the widget (pick the session, then **Save session note**; one note per session, saving replaces it).

## 5. Close the connection

In [ ]:
conn.close()
print("OMERO connection closed.")